In [1]:
from SPARQLWrapper import SPARQLWrapper, JSON
import pandas as pd

sparql = SPARQLWrapper("https://query.wikidata.org/sparql")
sparql.setReturnFormat(JSON)
sparql.addCustomHttpHeader(
    "User-Agent",
    "MyResearchProject/1.0 (your_real_email@example.com)"
)
query = """
SELECT ?company ?companyLabel ?companyId ?exchangeLabel
       (SAMPLE(?ticker0) AS ?ticker)
       (SAMPLE(?industryLabel0) AS ?industry)
       (SAMPLE(?marketCap0) AS ?marketCap)
       (SAMPLE(?marketCapDate0) AS ?marketCapDate)
       (SAMPLE(?revenue0) AS ?revenue)
       (SAMPLE(?revenueDate0) AS ?revenueDate)
WHERE {
  VALUES ?targetExchange { wd:Q13677 wd:Q82059 }   # NYSE, Nasdaq

  ?company p:P414 ?listingStatement .
  ?listingStatement ps:P414 ?targetExchange .

  OPTIONAL { ?listingStatement pq:P249 ?ticker0 . }

  OPTIONAL {
    ?company wdt:P452 ?industry0 .
    ?industry0 rdfs:label ?industryLabel0 .
    FILTER(LANG(?industryLabel0) = "en")
  }

  OPTIONAL {
    {
      SELECT ?company ?marketCap0 ?marketCapDate0 WHERE {
        {
          SELECT ?company (MAX(?capDate) AS ?marketCapDate0) WHERE {
            ?company p:P2226 ?capStmt .
            ?capStmt pq:P585 ?capDate .
          }
          GROUP BY ?company
        }

        ?company p:P2226 ?capStmt .
        ?capStmt pq:P585 ?marketCapDate0 ;
                 psv:P2226 ?capValue .

        ?capValue wikibase:quantityAmount ?marketCap0 .
      }
    }
  }
  OPTIONAL {
  {
    SELECT ?company ?revenue0 ?revenueDate0 WHERE {
      {
        SELECT ?company (MAX(?revDate) AS ?revenueDate0) WHERE {
          ?company p:P2139 ?revStmt .
          ?revStmt pq:P585 ?revDate .
        }
        GROUP BY ?company
      }

      ?company p:P2139 ?revStmt .
      ?revStmt pq:P585 ?revenueDate0 ;
               psv:P2139 ?revValue .

      ?revValue wikibase:quantityAmount ?revenue0 .
    }
  }
}

  BIND(REPLACE(STR(?company), "http://www.wikidata.org/entity/", "") AS ?companyId)

  SERVICE wikibase:label {
    bd:serviceParam wikibase:language "[AUTO_LANGUAGE],mul,en".
    ?company rdfs:label ?companyLabel .
    ?targetExchange rdfs:label ?exchangeLabel .
  }
}
GROUP BY ?company ?companyLabel ?companyId ?exchangeLabel
ORDER BY ?companyLabel
LIMIT 10000
"""

sparql.setQuery(query)
results = sparql.query().convert()

rows = []
for row in results["results"]["bindings"]:
    rows.append({
        "company": row.get("companyLabel", {}).get("value"),
        "company_id": row.get("companyId", {}).get("value"),
        "exchange": row.get("exchangeLabel", {}).get("value"),
        "ticker": row.get("ticker", {}).get("value"),
        "industry": row.get("industry", {}).get("value"),
        "market_cap": row.get("marketCap", {}).get("value"),
        "market_cap_date": row.get("marketCapDate", {}).get("value"),
        "company_url": row.get("company", {}).get("value"),
        "revenue": row.get("revenue", {}).get("value"),
        "revenue_date": row.get("revenueDate", {}).get("value"),
    })

df = pd.DataFrame(rows)

# optional cleanup
df["market_cap"] = pd.to_numeric(df["market_cap"], errors="coerce")
df["market_cap_date"] = pd.to_datetime(df["market_cap_date"], errors="coerce")
df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
df["revenue_date"] = pd.to_datetime(df["revenue_date"], errors="coerce")

df = df.sort_values(
    ["revenue", "company"],
    ascending=[False, True],
    na_position="last"
).reset_index(drop=True)

df.to_csv("nyse_nasdaq_companies_all.csv", index=False, encoding="utf-8")
print(df.head(20))
print("Total rows:", len(df))

                                 company company_id                 exchange  \
0                                  Honda      Q9584  New York Stock Exchange   
1                                 Nissan     Q20165                   Nasdaq   
2                                 Itochu    Q717093                   Nasdaq   
3                             Sony Group     Q41187  New York Stock Exchange   
4                             NTT DoCoMo    Q853958  New York Stock Exchange   
5                             Canon Inc.     Q62621  New York Stock Exchange   
6              Micro Focus International   Q1931458  New York Stock Exchange   
7                                Sinopec    Q831445  New York Stock Exchange   
8             PetroChina Company Limited    Q503182  New York Stock Exchange   
9                                   TSMC    Q713418  New York Stock Exchange   
10                          Nomura Group    Q658089  New York Stock Exchange   
11                         América Móvil

In [ ]:
# Convert numeric/date columns
df["market_cap"] = pd.to_numeric(df["market_cap"], errors="coerce")
df["market_cap_date"] = pd.to_datetime(df["market_cap_date"], errors="coerce")
df["revenue"] = pd.to_numeric(df["revenue"], errors="coerce")
df["revenue_date"] = pd.to_datetime(df["revenue_date"], errors="coerce")

# Clean IDs
df["company_id"] = df["company_id"].astype(str).str.strip()

# Sort but keep missing revenue
df = df.sort_values(
    ["revenue", "company"],
    ascending=[False, True],
    na_position="last"
).reset_index(drop=True)

# Save raw version before deduplication

def first_non_null(series):
    series = series.dropna()
    return series.iloc[0] if len(series) > 0 else pd.NA


def join_unique(series):
    values = series.dropna().astype(str).str.strip()
    values = values[values != ""]
    return "; ".join(sorted(set(values)))


# Deduplicate by Wikidata company ID
df_clean = (
    df.groupby("company_id", as_index=False)
      .agg({
          "company": first_non_null,
          "exchange": join_unique,
          "ticker": join_unique,
          "industry": first_non_null,
          "market_cap": first_non_null,
          "market_cap_date": first_non_null,
          "company_url": first_non_null,
          "revenue": first_non_null,
          "revenue_date": first_non_null,
      })
)

# Primary ticker for later market-cap retrieval
df_clean["primary_ticker"] = df_clean["ticker"].str.split(";").str[0].str.strip()

# Save final cleaned company universe

print("Raw rows:", len(df))
print("Unique companies:", len(df_clean))
print("Companies with revenue:", df_clean["revenue"].notna().sum())
print("Companies with Wikidata market cap:", df_clean["market_cap"].notna().sum())
print("Companies with industry:", df_clean["industry"].notna().sum())

df_clean.head(20)

Raw rows: 4263
Unique companies: 4184
Companies with revenue: 602
Companies with Wikidata market cap: 213
Companies with industry: 2966


,company_id,company,exchange,ticker,industry,market_cap,market_cap_date,company_url,revenue,revenue_date,primary_ticker
0,Q1001788,Buenaventura,New York Stock Exchange,BVN,mining,<NA>,<NA>,http://www.wikidata.org/entity/Q1001788,<NA>,<NA>,BVN
1,Q1002992,Build-A-Bear Workshop,New York Stock Exchange,BBW,retail,<NA>,<NA>,http://www.wikidata.org/entity/Q1002992,<NA>,<NA>,BBW
2,Q100321332,United Nuclear Corporation,New York Stock Exchange,UNC,mining,<NA>,<NA>,http://www.wikidata.org/entity/Q100321332,<NA>,<NA>,UNC
3,Q100323973,Whitestone REIT,New York Stock Exchange,WSR,NaN,<NA>,<NA>,http://www.wikidata.org/entity/Q100323973,<NA>,<NA>,WSR
4,Q1007000,Genpact,New York Stock Exchange,G,professional service,<NA>,<NA>,http://www.wikidata.org/entity/Q1007000,<NA>,<NA>,G
5,Q1009458,Bunge Limited,New York Stock Exchange,BG,food industry,<NA>,<NA>,http://www.wikidata.org/entity/Q1009458,67232000000.0,2022-01-01 00:00:00+00:00,BG
6,Q101209811,Pool Corporation,New York Stock Exchange,POOL,NaN,<NA>,<NA>,http://www.wikidata.org/entity/Q101209811,<NA>,<NA>,POOL
7,Q101209850,Vontier,New York Stock Exchange,VNT,NaN,<NA>,<NA>,http://www.wikidata.org/entity/Q101209850,<NA>,<NA>,VNT
8,Q101429880,Adecoagro,New York Stock Exchange,AGRO,agribusiness,<NA>,<NA>,http://www.wikidata.org/entity/Q101429880,<NA>,<NA>,AGRO
9,Q101606525,Aurora Innovation,Nasdaq,AUR,autonomous car,<NA>,<NA>,http://www.wikidata.org/entity/Q101606525,<NA>,<NA>,AUR


In [7]:
df_clean.to_csv("nyse_nasdaq_companies_all.csv", index=False, encoding="utf-8")
